In [86]:
import numpy as np
import pandas as pd
import re
import nltk
import string 
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import unicodedata
from nltk.corpus import words
string.punctuation
nltk.download('words')
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')
porter = PorterStemmer()
english_words = set(words.words())

[nltk_data] Downloading package words to
[nltk_data]     C:\Users\imerp\AppData\Roaming\nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\imerp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\imerp\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\imerp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [87]:
#Read CSV
data = pd.read_csv('sample1.csv')
text_data = data[['issue_title','issue_body']]


print(text_data)

                                             issue_title  \
0      __tileSrcRect is null in Entities.ldtk Sample ...   
1                   Update the blog link in the SEF Site   
2      🐛 Parser cannot properly distinguish between p...   
3      "Row added" snackbar is not visible if the gri...   
4      [CRASH]  _tcp_write_on_socket crashes when flu...   
...                                                  ...   
72284  [HELP] Volume Mount hostpath: Unable to mount ...   
72285        is form.progress valid for upload progress?   
72286  RPi used for door contact sensors, warning in log   
72287  Tail latency of envoy proxy is bad if every re...   
72288  Setting the date value programmatically  after...   

                                              issue_body  
0      In the Entities example, we there are some `__...  
1      **Describe the bug**\r\nUpdate the blog link i...  
2      Consider these two expressions:\r\n```\r\nf (g...  
3      ## Description  \r\nWhen grid has no

In [88]:
def features_eng(text):
    
    question_count = len(re.findall(r'\?', text))
    
    
    return question_count
    
     
text_data['questions_title']  = text_data['issue_title'].astype(str).apply(features_eng)
text_data['questions_body'] = text_data['issue_body'].astype(str).apply(features_eng)

C:\Users\imerp\AppData\Local\Temp\ipykernel_9788\882056174.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  text_data['questions_title']  = text_data['issue_title'].astype(str).apply(features_eng)
C:\Users\imerp\AppData\Local\Temp\ipykernel_9788\882056174.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  text_data['questions_body'] = text_data['issue_body'].astype(str).apply(features_eng)


In [89]:
#Text Preprocessing
def text_preprocessing(text):

        
    
        if not isinstance(text, str):
            text = ""

        text = text.lower() #Lower case

        #Delete emojis and non-alfabetical
        text = unicodedata.normalize('NFD', text)

        text = re.sub(r'\[.*?\]', '', text).strip()
    
        text = re.sub(r'[\u0300-\u036f]', '', text)
        
        text = re.sub(r'[^a-zA-Z\s.,!?\'"-]', '', text)

        text = re.sub(r'\b\w\b', '', text)

        text = re.sub(r'\s+', ' ', text).strip() #Delete \n \r...
        
        text = "".join([i for i in text if i not in string.punctuation]) #Punctuation free

        
        tokens = nltk.word_tokenize(text) #Text tokenize
    
        
    
        stop_words = set(stopwords.words('english')) #Stop words removal
    
        tokens = [word for word in tokens if word not in stop_words]

        tokens = [word for word in tokens if word in english_words] #Delete non-existing words

        tokens = [porter.stem(word) for word in tokens] #Stemming

        word_count = len(tokens)

        return tokens, word_count
    

In [90]:
#Do text prepocessing
text_data[['token_title','count_words_title']] = text_data['issue_title'].astype(str).apply(text_preprocessing).apply(pd.Series)
text_data[['token_body','count_words_body']] = text_data['issue_body'].astype(str).apply(text_preprocessing).apply(pd.Series)
text_data['questions'] = text_data['questions_title'] + text_data['questions_body']
text_data['issue_label'] =  data['issue_label']
text_data = text_data.drop(['questions_title','questions_body'],axis=1)
text_data = text_data.dropna()
print(text_data.head())
text_data.to_csv('text.csv')


C:\Users\imerp\AppData\Local\Temp\ipykernel_9788\2845124564.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  text_data[['token_title','count_words_title']] = text_data['issue_title'].astype(str).apply(text_preprocessing).apply(pd.Series)
C:\Users\imerp\AppData\Local\Temp\ipykernel_9788\2845124564.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  text_data[['token_title','count_words_title']] = text_data['issue_title'].astype(str).apply(text_preprocessing).apply(pd.Series)


                                         issue_title  \
0  __tileSrcRect is null in Entities.ldtk Sample ...   
1               Update the blog link in the SEF Site   
2  🐛 Parser cannot properly distinguish between p...   
3  "Row added" snackbar is not visible if the gri...   
4  [CRASH]  _tcp_write_on_socket crashes when flu...   

                                          issue_body  \
0  In the Entities example, we there are some `__...   
1  **Describe the bug**\r\nUpdate the blog link i...   
2  Consider these two expressions:\r\n```\r\nf (g...   
3  ## Description  \r\nWhen grid has no height an...   
4  <!--\r\nThank you for reporting a crash in Ope...   

                                token_title  count_words_title  \
0    [null, sampl, nullabl, accord, schema]                  5   
1                       [updat, link, site]                  3   
2  [parser, properli, distinguish, multipl]                  4   
3    [row, ad, visibl, grid, small, height]                  6